# SQL Testing with Assertions

## Overview
SQL testing is built on a simple convention: **a test is a query that returns rows only when it fails**. Zero rows = pass. Any rows returned = the test failed, and those rows are the evidence.

This pattern requires no testing framework — just a SQL client, a Python script, or a CI runner that checks whether the query returns rows.

**The assertion template:**
```sql
-- TEST: no_negative_premiums
-- Returns: rows on FAILURE (each row is a violation)
SELECT policy_id, premium
FROM   policies
WHERE  premium < 0;
-- 0 rows → PASS   |   1+ rows → FAIL
```

**Dataset:** insurance + healthcare tables with intentionally seeded data quality issues (negative premiums, duplicate lab results, impossible values, null required fields) so every category of test has something meaningful to catch.

---

In [ ]:
import sqlite3, json
from datetime import date, timedelta

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

conn.executescript("""
-- Mixed dataset: insurance + healthcare + financial transactions
CREATE TABLE customers (
    customer_id  TEXT PRIMARY KEY,
    full_name    TEXT NOT NULL,
    email        TEXT,
    province     TEXT NOT NULL,
    segment      TEXT NOT NULL,   -- 'retail','commercial','group'
    created_at   TEXT NOT NULL
);

CREATE TABLE policies (
    policy_id    TEXT PRIMARY KEY,
    customer_id  TEXT NOT NULL REFERENCES customers(customer_id),
    product_line TEXT NOT NULL,   -- 'auto','home','life','health'
    premium      REAL NOT NULL,
    status       TEXT NOT NULL,   -- 'active','lapsed','cancelled'
    start_date   TEXT NOT NULL,
    end_date     TEXT
);

CREATE TABLE claims (
    claim_id     TEXT PRIMARY KEY,
    policy_id    TEXT NOT NULL REFERENCES policies(policy_id),
    claim_date   TEXT NOT NULL,
    amount       REAL NOT NULL,
    status       TEXT NOT NULL,   -- 'open','approved','denied','paid'
    approved_by  TEXT
);

CREATE TABLE lab_results (
    result_id    TEXT PRIMARY KEY,
    patient_id   TEXT NOT NULL,
    test_name    TEXT NOT NULL,
    result_value REAL,
    unit         TEXT,
    result_date  TEXT NOT NULL,
    flagged      INTEGER DEFAULT 0
);

-- Intentionally seeded with some data quality issues for demos
""")

# Clean customers
customers = [
    ('C001','Alice Nguyen',    'alice@email.com',  'ON','retail',    '2022-01-15'),
    ('C002','Bob Harrington',  'bob@email.com',    'BC','commercial','2022-03-01'),
    ('C003','Fatima Al-Rashid','fatima@email.com', 'QC','group',     '2022-06-10'),
    ('C004','James MacLeod',   'james@email.com',  'NS','retail',    '2023-01-20'),
    ('C005','Mei-Ling Chen',   'mei@email.com',    'AB','commercial','2023-04-05'),
    ('C006','David Park',      None,               'ON','retail',    '2024-01-01'),  # null email
    ('C007','Sandra Okafor',   'sandra@email.com', 'ON','retail',    '2024-02-15'),
]
conn.executemany("INSERT INTO customers VALUES (?,?,?,?,?,?)", customers)

# Policies — including some with data issues
policies = [
    ('POL-001','C001','auto',  1200.0,'active',   '2023-01-15','2024-01-15'),
    ('POL-002','C001','home',  1800.0,'active',   '2023-03-01','2024-03-01'),
    ('POL-003','C002','auto',  2400.0,'active',   '2022-06-01','2023-06-01'),
    ('POL-004','C003','life',   600.0,'lapsed',   '2022-09-01','2023-09-01'),
    ('POL-005','C004','auto',  1400.0,'active',   '2023-02-01','2024-02-01'),
    ('POL-006','C005','health',4800.0,'active',   '2023-05-01','2024-05-01'),
    ('POL-007','C006','home',  2100.0,'cancelled','2024-01-15','2024-06-15'),
    ('POL-008','C007','auto',  1100.0,'active',   '2024-03-01','2025-03-01'),
    ('POL-009','C002','home',  -500.0,'active',   '2024-01-01','2025-01-01'),  # negative premium (data issue)
    ('POL-010','C004','auto',  1400.0,'active',   '2023-02-01','2024-02-01'),  # duplicate of POL-005
]
conn.executemany("INSERT INTO policies VALUES (?,?,?,?,?,?,?)", policies)

# Claims
claims = [
    ('CLM-001','POL-001','2023-06-15',2200.0,'paid',    'Dr. Lee'),
    ('CLM-002','POL-003','2023-08-01', 500.0,'approved','Dr. Pham'),
    ('CLM-003','POL-005','2023-11-20',8500.0,'paid',    'Dr. Singh'),
    ('CLM-004','POL-006','2024-01-10', 350.0,'open',    None),
    ('CLM-005','POL-001','2024-02-28',1100.0,'denied',  'Dr. Lee'),
    ('CLM-006','POL-008','2024-04-15', 750.0,'approved','Dr. Pham'),
    ('CLM-007','POL-003','2024-05-01',99999.0,'open',   None),  # suspiciously large
]
conn.executemany("INSERT INTO claims VALUES (?,?,?,?,?,?)", claims)

# Lab results — with some quality issues
labs = [
    ('LAB-001','P001','HbA1c',    7.2,'%',        '2023-10-01',1),
    ('LAB-002','P001','eGFR',    68.0,'mL/min',   '2023-10-01',0),
    ('LAB-003','P002','HbA1c',    8.4,'%',        '2024-01-10',1),
    ('LAB-004','P002','Creatinine',115.0,'umol/L','2024-01-10',1),
    ('LAB-005','P003','HbA1c',    None,'%',        '2024-02-15',0),  # null value
    ('LAB-006','P001','HbA1c',    7.2,'%',        '2023-10-01',1),  # duplicate of LAB-001
    ('LAB-007','P004','eGFR',   -5.0,'mL/min',   '2024-03-01',0),  # impossible negative
    ('LAB-008','P003','HbA1c',    9.1,'%',        '2024-02-15',1),
]
conn.executemany("INSERT INTO lab_results VALUES (?,?,?,?,?,?,?)", labs)
conn.commit()

print("Testing dataset loaded:")
for tbl in ['customers','policies','claims','lab_results']:
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:<15s}: {n} rows")
print()
print("Known data quality issues seeded:")
issues = [
    ("customers",    "C006 has NULL email"),
    ("policies",     "POL-009 has negative premium (-500)"),
    ("policies",     "POL-010 is a near-duplicate of POL-005"),
    ("claims",       "CLM-007 has suspiciously large amount (99999)"),
    ("lab_results",  "LAB-005 has NULL result_value"),
    ("lab_results",  "LAB-006 is an exact duplicate of LAB-001"),
    ("lab_results",  "LAB-007 has impossible negative eGFR (-5)"),
]
for tbl, issue in issues:
    print(f"  [{tbl}] {issue}")


---
## The assertion pattern: zero rows = pass

In [ ]:
print("=== SQL assertion pattern: zero-row = pass ===")
print()

print("Core principle: a test is a SQL query that returns rows only on FAILURE.")
print("  0 rows returned → test passes")
print("  1+ rows returned → test fails (the returned rows are the failing records)")
print()

def run_test(name, sql, conn):
    rows = conn.execute(sql).fetchall()
    status = "PASS ✓" if len(rows) == 0 else f"FAIL ✗  ({len(rows)} failing rows)"
    print(f"  [{status}] {name}")
    if rows:
        for r in rows[:3]:
            print(f"           → {dict(r)}")
    return len(rows) == 0

print("Running tests on policies table:")
print()

# Test 1: no negative premiums
run_test(
    "no_negative_premiums",
    "SELECT policy_id, premium FROM policies WHERE premium < 0",
    conn
)

# Test 2: status in accepted values
run_test(
    "policy_status_valid",
    "SELECT policy_id, status FROM policies WHERE status NOT IN ('active','lapsed','cancelled')",
    conn
)

# Test 3: no null customer_id in policies
run_test(
    "policy_customer_id_not_null",
    "SELECT policy_id FROM policies WHERE customer_id IS NULL",
    conn
)

# Test 4: all policy customer_ids exist in customers
run_test(
    "policy_customer_referential_integrity",
    """SELECT p.policy_id, p.customer_id
       FROM   policies p
       LEFT JOIN customers c ON c.customer_id = p.customer_id
       WHERE  c.customer_id IS NULL""",
    conn
)

# Test 5: end_date after start_date
run_test(
    "policy_end_after_start",
    "SELECT policy_id, start_date, end_date FROM policies WHERE end_date <= start_date",
    conn
)

print()
print("PostgreSQL equivalent (same pattern):")
print("""
  -- In a test script or CI pipeline:
  DO $$
  DECLARE v_count INT;
  BEGIN
      SELECT COUNT(*) INTO v_count
      FROM policies WHERE premium < 0;
      ASSERT v_count = 0,
             FORMAT('no_negative_premiums FAILED: %s rows', v_count);
  END $$;
""")


---
## Row count and completeness checks

In [ ]:
print("=== Row count and completeness checks ===")
print()

import datetime

def run_test(name, sql, conn):
    rows = conn.execute(sql).fetchall()
    status = "PASS ✓" if len(rows) == 0 else f"FAIL ✗  ({len(rows)} failing rows)"
    print(f"  [{status}] {name}")
    for r in rows[:3]:
        print(f"           → {dict(r)}")
    return len(rows) == 0

def run_value_test(name, sql, expected, conn):
    result = conn.execute(sql).fetchone()[0]
    status = "PASS ✓" if result == expected else f"FAIL ✗  (got {result}, expected {expected})"
    print(f"  [{status}] {name}")
    return result == expected

print("Row count tests:")
# Minimum row count
run_value_test(
    "customers_minimum_count",
    "SELECT COUNT(*) FROM customers",
    expected=7, conn=conn
)

# No empty tables
for tbl in ['customers','policies','claims','lab_results']:
    rows = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    status = "PASS ✓" if rows > 0 else "FAIL ✗  (empty table!)"
    print(f"  [{status}] {tbl}_not_empty  ({rows} rows)")

print()
print("Completeness (NOT NULL) tests:")
# Required columns
run_test("customers_full_name_not_null",
         "SELECT customer_id FROM customers WHERE full_name IS NULL OR TRIM(full_name)=''", conn)
run_test("customers_province_not_null",
         "SELECT customer_id FROM customers WHERE province IS NULL", conn)
run_test("lab_results_value_not_null",
         "SELECT result_id, test_name FROM lab_results WHERE result_value IS NULL", conn)
run_test("claims_amount_positive",
         "SELECT claim_id, amount FROM claims WHERE amount <= 0", conn)

print()
print("Freshness check (data recency):")
# Check most recent record isn't too old
most_recent = conn.execute("SELECT MAX(created_at) FROM customers").fetchone()[0]
cutoff = '2023-01-01'
status = "PASS ✓" if most_recent >= cutoff else f"FAIL ✗  (most recent: {most_recent})"
print(f"  [{status}] customers_freshness  (most recent: {most_recent})")

print()
print("PostgreSQL freshness check:")
print("""
  -- Alert if no new policies in last 24 hours:
  SELECT CASE
      WHEN MAX(created_at) < NOW() - INTERVAL '24 hours'
      THEN 'STALE: no new data in 24h'
      ELSE 'FRESH'
  END AS freshness_status
  FROM policies;
""")


---
## Constraint validation and uniqueness tests

In [ ]:
print("=== Constraint validation and uniqueness tests ===")
print()

def run_test(name, sql, conn):
    rows = conn.execute(sql).fetchall()
    status = "PASS ✓" if len(rows) == 0 else f"FAIL ✗  ({len(rows)} failing rows)"
    print(f"  [{status}] {name}")
    for r in rows[:3]:
        print(f"           → {dict(r)}")
    return len(rows) == 0

print("Uniqueness tests:")
# Primary key uniqueness (should always pass if PK constraint is enforced)
run_test("customers_customer_id_unique",
    """SELECT customer_id, COUNT(*) AS n
       FROM customers GROUP BY customer_id HAVING COUNT(*) > 1""", conn)

# Near-duplicate detection (same customer + product + dates)
run_test("policies_no_duplicates_same_customer_product_dates",
    """SELECT customer_id, product_line, start_date, COUNT(*) AS n
       FROM   policies
       GROUP  BY customer_id, product_line, start_date
       HAVING COUNT(*) > 1""", conn)

# Exact duplicate rows in lab_results
run_test("lab_results_no_exact_duplicates",
    """SELECT patient_id, test_name, result_date, COUNT(*) AS n
       FROM   lab_results
       GROUP  BY patient_id, test_name, result_date
       HAVING COUNT(*) > 1""", conn)

print()
print("Domain / range checks:")
run_test("lab_results_egfr_non_negative",
    """SELECT result_id, test_name, result_value
       FROM   lab_results
       WHERE  test_name = 'eGFR' AND result_value < 0""", conn)

run_test("lab_results_hba1c_plausible_range",
    """SELECT result_id, result_value
       FROM   lab_results
       WHERE  test_name = 'HbA1c'
         AND  result_value IS NOT NULL
         AND  (result_value < 3.0 OR result_value > 20.0)""", conn)

run_test("claims_amount_not_extreme",
    """SELECT claim_id, amount FROM claims WHERE amount > 50000""", conn)

print()
print("Accepted values tests:")
run_test("customers_segment_accepted_values",
    """SELECT customer_id, segment FROM customers
       WHERE segment NOT IN ('retail','commercial','group')""", conn)

run_test("claims_status_accepted_values",
    """SELECT claim_id, status FROM claims
       WHERE status NOT IN ('open','approved','denied','paid')""", conn)

run_test("policies_product_line_accepted_values",
    """SELECT policy_id, product_line FROM policies
       WHERE product_line NOT IN ('auto','home','life','health')""", conn)


---
## Common Pitfalls

**1. Writing tests that return rows on SUCCESS instead of FAILURE**
The SQL testing convention is: return rows = FAIL; zero rows = PASS. `SELECT COUNT(*) FROM policies WHERE premium > 0` always returns a row (the count). The correct test is `SELECT policy_id FROM policies WHERE premium <= 0` — which returns rows only when there is a violation.

**2. Testing only the happy path**
Testing that valid data passes is necessary but not sufficient. Also test that invalid data is caught: insert a row with a NULL required field, a negative premium, or an invalid status, and verify the test catches it. A test that never fails is not a reliable test.

**3. Skipping referential integrity tests**
Database FK constraints are often disabled in analytical databases for performance. Without FK constraints, orphan rows accumulate silently. Explicit referential integrity tests with LEFT JOIN ... WHERE right.key IS NULL are the analytical DB equivalent of FK enforcement.

**4. Not including the failing row in the test output**
A test that returns only a COUNT tells you a test failed but not which rows are bad. Always SELECT the identifying columns (policy_id, claim_id) and the offending values so failures can be diagnosed without running a separate query.

**5. Hardcoding expected counts that change with data growth**
`SELECT CASE WHEN COUNT(*) = 7 THEN 'pass' END FROM customers` will fail the moment a legitimate new customer is added. Use minimum-count tests (`WHERE COUNT(*) < 5`) or structure-based tests, not exact counts on live data.

**6. Not testing after every ETL load**
Running tests only manually or on a weekly schedule means data quality issues accumulate for days before detection. Hook tests into the ETL pipeline: run the full test suite after every load and alert on failures before downstream consumers read the data.


---
*sql_methods_library - Samantha McGarrigle*